# Keras-RL, DQN y la evolución de las arquitecturas de aprendizaje por refuerzo

- Artículo completo en la plataforma: https://fuzzyfrog.ai/es/ai-lab/proyectos/educacion/keras-rl-dqn-arquitecturas-aplicaciones-aprendizaje-refuerzo/
- Este notebook entrena un agente DQN, con Keras-RL sobre TensorFlow y Keras, para jugar Space Invaders de Atari a partir únicamente de los frames de video del juego.
- Requiere un entorno local para poder correr las dependencias clásicas de Gym y Atari. Ver la sección de instalación.

## Diagrama de arquitectura

`Frame del juego → preprocesamiento (escala de grises + apilado) → red convolucional (3 capas conv + densa) → política de exploración decreciente → memoria de repetición + red objetivo → entrenamiento → evaluación`

Cada pieza de este pipeline existe para que el agente aprenda de su propia experiencia sin desestabilizarse en el camino, no para decirle qué hacer.

## Instalación de dependencias

**Nota:** por incompatibilidades de renderizado en notebooks alojados en la nube, este proyecto se ejecuta como scripts independientes (`%%writefile` + `!python`) en lugar de celdas encadenadas. Es una fricción real del entorno de trabajo, documentada en el artículo.

In [ ]:
# 1. Crear un entorno virtual para aislar las versiones exactas de las dependencias
!pip install virtualenv --quiet
!virtualenv miar_rl

# 2. Instalar las versiones compatibles entre si dentro del entorno virtual
!./miar_rl/bin/pip install numpy==1.23.5 --quiet
!./miar_rl/bin/pip install gym==0.17.3 --quiet
!./miar_rl/bin/pip install tensorflow==2.12.1 keras==2.12.0 --quiet
!./miar_rl/bin/pip install git+https://github.com/Kojoley/atari-py.git@1.2.2 --quiet
!./miar_rl/bin/pip install keras-rl2==1.0.5 --quiet
!./miar_rl/bin/pip install Pillow matplotlib --quiet

## Creación del entorno

Se crea el entorno de Atari Space Invaders sobre la interfaz clásica de Gym, y se revisan sus dimensiones y acciones posibles.

In [ ]:
%%writefile cell_entorno.py
import numpy as np
import gym

env_name = 'SpaceInvaders-v0'
env = gym.make(env_name)

np.random.seed(123)
env.seed(123)
nb_actions = env.action_space.n

print(f"Entorno {env_name} creado con nb_actions = {nb_actions}")
print("El tamano de nuestro frame es: ", env.observation_space)
print("Las acciones posibles son: ", env.env.get_action_meanings())

In [ ]:
!./miar_rl/bin/python cell_entorno.py

## Explicación de datos: preprocesamiento de frames

Cada frame llega en color, a resolución de 210 por 160 píxeles. Se convierte a escala de grises y se reduce a 84 por 84, para acelerar el entrenamiento sin perder la información visual relevante para decidir la acción.

In [ ]:
%%writefile cell_procesador.py
from cell_entorno import *
from rl.core import Processor
from PIL import Image

INPUT_SHAPE = (84, 84)

class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3  # (height, width, channel)
        img = Image.fromarray(observation)
        img = img.resize(INPUT_SHAPE).convert('L')
        processed_observation = np.array(img)
        assert processed_observation.shape == INPUT_SHAPE
        return processed_observation.astype('uint8')

    def process_state_batch(self, batch):
        return batch.astype('float32') / 255.0

    def process_reward(self, reward):
        return np.clip(reward, -1., 1.)

In [ ]:
!./miar_rl/bin/python cell_procesador.py

## Modelado: arquitectura de la red y del agente DQN

Tres capas convolucionales extraen los patrones visuales del frame apilado, seguidas de una capa de aplanamiento y una capa densa que estima el valor de cada acción posible. El agente combina esa red con una memoria de repetición, una política de exploración decreciente y una red objetivo que se actualiza de forma periódica.

In [ ]:
%%writefile main.py
import numpy as np
import gym
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Permute
from tensorflow.keras.optimizers import legacy as keras_legacy_optimizers
from rl.agents.dqn import DQNAgent
from rl.memory import SequentialMemory
from rl.policy import LinearAnnealedPolicy, EpsGreedyQPolicy
from rl.core import Processor

# Parametros
ENV_NAME = 'SpaceInvaders-v0'
INPUT_SHAPE = (84, 84)
WINDOW_LENGTH = 4
SEED = 123

# ------------------------------------------------------------
# Procesador personalizado
# ------------------------------------------------------------
class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3
        img = Image.fromarray(observation).resize(INPUT_SHAPE).convert('L')
        return np.array(img).astype('uint8')

    def process_state_batch(self, batch):
        return batch.astype('float32') / 255.0

    def process_reward(self, reward):
        return np.clip(reward, -1., 1.)

# ------------------------------------------------------------
# Clase contenedora del agente
# ------------------------------------------------------------
class DQNAgentWrapper:
    def __init__(self, env_name):
        self.env_name = env_name
        self.env = gym.make(env_name)
        self.env.seed(SEED)
        np.random.seed(SEED)
        tf.random.set_seed(SEED)

        self.nb_actions = self.env.action_space.n
        self.processor = AtariProcessor()
        self.model = self.build_model()
        self.memory = SequentialMemory(limit=100000, window_length=WINDOW_LENGTH)
        self.policy = LinearAnnealedPolicy(
            EpsGreedyQPolicy(),
            attr='eps',
            value_max=1.0,
            value_min=0.1,
            value_test=0.05,
            nb_steps=100000  # decaimiento lento, para explorar mas antes de confiar en la red
        )
        self.agent = self.build_agent()

    def build_model(self):
        model = Sequential()
        model.add(Permute((2, 3, 1), input_shape=(WINDOW_LENGTH,) + INPUT_SHAPE))
        model.add(Conv2D(32, (8, 8), strides=(4, 4), activation='relu'))
        model.add(Conv2D(64, (4, 4), strides=(2, 2), activation='relu'))
        model.add(Conv2D(64, (3, 3), activation='relu'))
        model.add(Flatten())
        model.add(Dense(512, activation='relu'))
        model.add(Dense(self.nb_actions, activation='linear'))
        return model

    def build_agent(self):
        adam_legacy = keras_legacy_optimizers.Adam(learning_rate=0.0001)
        dqn = DQNAgent(
            model=self.model,
            nb_actions=self.nb_actions,
            memory=self.memory,
            processor=self.processor,
            policy=self.policy,
            nb_steps_warmup=20000,      # warmup mas largo, mas experiencia antes de entrenar
            gamma=0.99,
            target_model_update=500,    # sincronizacion mas frecuente, mas estabilidad
            train_interval=2,           # entrena cada 2 pasos, en vez de cada 4
            delta_clip=1.0
        )
        dqn.compile(adam_legacy, metrics=['mae'])
        return dqn

    def train(self, steps=50000):
        self.agent.fit(self.env, nb_steps=steps, visualize=False, verbose=2)

    def test(self, episodes=3, render=False):
        self.agent.test(self.env, nb_episodes=episodes, visualize=render)

    def save(self, path='dqn_weights.h5f'):
        self.agent.save_weights(path, overwrite=True)

    def load(self, path='dqn_weights.h5f'):
        self.agent.load_weights(path)

if __name__ == '__main__':
    dqn_wrapper = DQNAgentWrapper(ENV_NAME)

    print("Entrenando el agente...")
    dqn_wrapper.train()

    print("Evaluando el agente...")
    dqn_wrapper.test()

    print("Guardando pesos...")
    dqn_wrapper.save(f'dqn_{ENV_NAME}_weights.h5f')

## Evaluación

El criterio de éxito del ejercicio es una recompensa promedio superior a 20 puntos en modo de prueba. La celda de test corre varios episodios con el agente ya entrenado y visualize desactivado, adecuado para correr sin interfaz gráfica.

In [ ]:
!./miar_rl/bin/python main.py

## Hallazgos principales

- Usar frames apilados en escala de grises, en lugar de un único frame a color, le da a la red información temporal suficiente para inferir movimiento sin disparar el costo de cómputo.
- Aumentar los pasos de calentamiento y reducir el intervalo de entrenamiento mejoró la estabilidad del aprendizaje frente a la primera versión del agente.
- Sincronizar la red objetivo con más frecuencia, junto con una tasa de aprendizaje más baja, priorizó estabilidad sobre velocidad de entrenamiento.
- El principal obstáculo no fue algorítmico, fue de entorno de ejecución. La incompatibilidad de renderizado en notebooks alojados en la nube obligó a mover cada bloque de entrenamiento a scripts independientes.

## Nota sobre la librería

Keras-RL cumplió su propósito y hoy ya no recibe mantenimiento activo. Quien quiera empezar un proyecto nuevo de aprendizaje por refuerzo hoy encontrará mejor soporte en librerías más recientes construidas sobre el mismo principio, con más algoritmos disponibles y compatibilidad con los entornos de simulación actuales. El código de este notebook se conserva tal cual, como registro fiel de cómo se entrenó este agente en su momento.